# Risk/UQ Paper Track: Risk Model Training

## Notebook Objective
This notebook builds the **candidate-level risk modeling pipeline** for CRAR (Calibrated Risk-Aware Reranking).
It is the training-stage notebook that transforms raw closed-loop candidate rollouts into calibrated probabilities for:
- `collision`
- `offroad`
- `failure_proxy`

This notebook is intended to produce reusable training artifacts for downstream benchmark and paper-export notebooks.


## Research Questions and Hypotheses

### Primary Questions
1. Can a compact ensemble model produce calibrated risk probabilities in closed-loop candidate rollouts?
2. Does calibration improve reliability without destroying discrimination?
3. Are learned risk estimates useful enough for control-time reranking?

### Hypotheses
- **H1 (Calibration):** temperature scaling reduces ECE/NLL relative to raw probabilities.
- **H2 (Signal):** ensemble epistemic variance increases on hard/high-interaction examples.
- **H3 (Control-readiness):** calibrated `failure_proxy_h15` scores can be used as stable reranking signals.


## Methodology

### Experimental Design
- Build candidate rows from LatentDriver action distributions.
- Generate short-horizon rollouts per candidate (`H_control=6`).
- Extract feature groups:
  - distribution statistics (`entropy`, mixture weights/std)
  - latent uncertainty traces (belief KL and predictive divergence summaries)
  - short-horizon dynamics/safety summaries (`progress`, `min_ttc`, `risk_sks`)
- Train deep-ensemble MLP (5 members by default) with multi-head binary targets for horizons `5/10/15`.
- Fit per-head temperature scalers on validation split.
- Fit conformal threshold on calibrated `failure_proxy_h15`.

### Outputs
- dataset artifacts (`*_risk_dataset*`)
- model artifacts (`*_risk_ensemble_member_*.npz`, metadata)
- calibration artifacts (`*_risk_temperature_scalers.json`, `*_risk_conformal_thresholds.json`)


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

REPO_ROOT = Path(r'/Users/achyutmorang/Downloads/waymax research/waymax-simulation-experiments')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.closedloop import ClosedLoopConfig
from src.workflows.risk_training_flow import run_risk_training_flow

cfg = ClosedLoopConfig()
cfg.run_prefix = str(REPO_ROOT / 'artifacts' / 'risk_uq_pilot')

# Pilot defaults (adjust for paper-scale run)
cfg.n_total_scenarios = 2400
cfg.n_eval_scenarios = 200
cfg.strict_min_eval = 200
cfg.risk_dataset_candidate_count = 8
cfg.risk_model_ensemble_size = 5
cfg.risk_dataset_control_horizon_steps = 6
cfg.risk_dataset_label_horizons = (5, 10, 15)
cfg.risk_control_fail_budget = 0.20

cfg


## Runtime Preparation

This notebook expects an initialized closed-loop runner that already loaded scenarios with raw simulator state retained for evaluation slices.

If needed, prepare `runner` with existing closed-loop utilities before running training flow.


In [ ]:
# Example runner setup (uncomment in Colab when needed):
# from src.closedloop.core import build_closedloop_runner_and_splits
# runner, data, reference_idx, candidate_eval_idx, eval_idx_all, eval_idx, reference_df, base_eval_openloop_df = build_closedloop_runner_and_splits(
#     cfg=cfg,
#     data_iter=None,
#     dataset_config=None,
#     n_shards=1,
#     shard_id=0,
# )


In [ ]:
# Execute training + calibration flow
# bundle = run_risk_training_flow(cfg=cfg, runner=runner)
# bundle.dataset_bundle.dataset_df.head()
# bundle.training_bundle.train_summary.head()
# bundle.calibration_bundle.summary_df


## Report Section Template: Training Findings

After execution, fill this section for manuscript-ready logging.

### Dataset Summary
- Number of candidate rows:
- Number of scenarios processed/skipped:
- Split composition (`train/val/test/high_interaction_holdout`):

### Model Quality (Validation)
- Raw NLL / Brier:
- Calibrated NLL / Brier:
- Best/worst heads by calibration:

### Calibration Narrative
- Which outputs improved after temperature scaling?
- Any heads where scaling stayed near `T=1.0`?
- Conformal threshold chosen for `failure_proxy_h15`:


In [ ]:
# Optional quick artifact check
# paths = bundle.artifact_paths
# for k, v in sorted(paths.items()):
#     print(f"{k}: {v}")


## Exit Criteria for This Notebook

Mark complete when all are true:
- [ ] Dataset artifacts written.
- [ ] Ensemble + metadata artifacts written.
- [ ] Temperature and conformal calibration artifacts written.
- [ ] Calibration summary table exported.
- [ ] Narrative notes filled in above.
